In [1]:
import pandas as pd
import geopandas as gpd
from unidecode import unidecode

## Add sentinel-2 data (GEE)

In [2]:
df = pd.read_csv("../data/raw/sentinel2_features_raw.csv", encoding='utf-8-sig')

In [3]:
df.head()

,province_name,year,season,s2_n_images,NDVI_mean,NDVI_min,NDVI_max,NDVI_range,EVI_mean,EVI_min,EVI_max,EVI_range,LSWI_mean,LSWI_min,LSWI_max,LSWI_range
0,An Giang,2015,autumn,1,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,Bà R?a?V?ng Tàu,2015,autumn,1,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,B?c Giang,2015,autumn,1,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,B?c K?n,2015,autumn,1,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,B?c Liêu,2015,autumn,1,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [4]:
df.columns

Index(['province_name', 'year', 'season', 's2_n_images', 'NDVI_mean',
       'NDVI_min', 'NDVI_max', 'NDVI_range', 'EVI_mean', 'EVI_min', 'EVI_max',
       'EVI_range', 'LSWI_mean', 'LSWI_min', 'LSWI_max', 'LSWI_range'],
      dtype='str')

In [6]:
# Check total missing values per column
missing_counts = df.isnull().sum()
print(missing_counts[missing_counts > 0]) # Shows only columns with missing data

# Or get a complete overview
df.info()

NDVI_mean     247
NDVI_min      247
NDVI_max      247
NDVI_range    247
EVI_mean      247
EVI_min       247
EVI_max       247
EVI_range     247
LSWI_mean     247
LSWI_min      247
LSWI_max      247
LSWI_range    247
dtype: int64
<class 'pandas.DataFrame'>
RangeIndex: 2048 entries, 0 to 2047
Data columns (total 16 columns):
 #   Column         Non-Null Count  Dtype  
---  ------         --------------  -----  
 0   province_name  2048 non-null   str    
 1   year           2048 non-null   int64  
 2   season         2048 non-null   str    
 3   s2_n_images    2048 non-null   int64  
 4   NDVI_mean      1801 non-null   float64
 5   NDVI_min       1801 non-null   float64
 6   NDVI_max       1801 non-null   float64
 7   NDVI_range     1801 non-null   float64
 8   EVI_mean       1801 non-null   float64
 9   EVI_min        1801 non-null   float64
 10  EVI_max        1801 non-null   float64
 11  EVI_range      1801 non-null   float64
 12  LSWI_mean      1801 non-null   float64
 13  LSWI_min  

In [7]:
df_all = df.copy()

In [8]:
# Count missing vegetation features per row
vegetation_cols = [
    "NDVI_mean", "NDVI_min", "NDVI_max", "NDVI_range",
    "EVI_mean", "EVI_min", "EVI_max", "EVI_range",
    "LSWI_mean", "LSWI_min", "LSWI_max", "LSWI_range",
]

missing_counts = df[vegetation_cols].isna().sum(axis=1)

print("Rows with all 12 features missing:",
      (missing_counts == len(vegetation_cols)).sum())

print("Rows with some features missing:",
      ((missing_counts > 0) &
       (missing_counts < len(vegetation_cols))).sum())

print("Rows with no missing vegetation features:",
      (missing_counts == 0).sum())

# Inspect rows missing all vegetation features
display(
    df.loc[
        missing_counts == len(vegetation_cols),
        ["province_name", "year", "season", "s2_n_images"]
    ].head(20)
)

Rows with all 12 features missing: 247
Rows with some features missing: 0
Rows with no missing vegetation features: 1801


,province_name,year,season,s2_n_images
0,An Giang,2015,autumn,1
1,Bà R?a?V?ng Tàu,2015,autumn,1
2,B?c Giang,2015,autumn,1
3,B?c K?n,2015,autumn,1
4,B?c Liêu,2015,autumn,1
5,B?c Ninh,2015,autumn,1
6,B?n Tre,2015,autumn,1
7,Bình ??nh,2015,autumn,1
8,Bình D??ng,2015,autumn,1
9,Bình Ph??c,2015,autumn,1


In [9]:
# Identify rows where all vegetation features are missing
missing_mask = df[vegetation_cols].isna().all(axis=1)

# Summarize by year
year_summary = (
    df.assign(veg_missing=missing_mask)
    .groupby("year")
    .agg(
        total_rows=("veg_missing", "size"),
        missing_rows=("veg_missing", "sum"),
    )
)

year_summary["complete_rows"] = (
    year_summary["total_rows"] - year_summary["missing_rows"]
)

year_summary["missing_pct"] = (
    year_summary["missing_rows"] / year_summary["total_rows"] * 100
).round(2)

display(year_summary)

#2. Find the first year with missing vegetation data
years_with_missing = year_summary[
    year_summary["missing_rows"] > 0
].index

if len(years_with_missing) > 0:
    print("First year with missing data:", years_with_missing.min())
    print("Years affected:", sorted(years_with_missing.tolist()))
else:
    print("No missing vegetation data found.")

,total_rows,missing_rows,complete_rows,missing_pct
year,,,,
2015,128,82,46,64.06
2016,192,84,108,43.75
2017,192,46,146,23.96
2018,192,35,157,18.23
2019,192,0,192,0.00
2020,192,0,192,0.00
2021,192,0,192,0.00
2022,192,0,192,0.00
2023,192,0,192,0.00


First year with missing data: 2015
Years affected: [2015, 2016, 2017, 2018]


In [10]:
vegetation_cols = [
    "NDVI_mean", "NDVI_min", "NDVI_max", "NDVI_range",
    "EVI_mean", "EVI_min", "EVI_max", "EVI_range",
    "LSWI_mean", "LSWI_min", "LSWI_max", "LSWI_range",
]

# Keep rows with complete vegetation features
df_veg = df.dropna(subset=vegetation_cols).copy()

print("Original dataset:", df.shape)
print("Vegetation-complete dataset:", df_veg.shape)
print("Rows removed:", len(df) - len(df_veg))

Original dataset: (2048, 16)
Vegetation-complete dataset: (1801, 16)
Rows removed: 247


In [16]:
# Apply unidecode directly to the pandas Series
df_veg['province_name'] = df_veg['province_name'].astype(str).apply(unidecode)

# Standardize any hyphens or spaces if needed
df_veg['province_name'] = df_veg['province_name'].str.strip()

In [22]:
print(df_veg['province_name'].unique())

<ArrowStringArray>
[        'Ha T?nh',      'Qu?ng Binh',        'An Giang',       'B?c Giang',
         'B?c K?n',        'B?c Ninh',         'B?n Tre',      'Binh D??ng',
      'Binh Ph??c',      'Binh Thu?n',         'C?n Th?',        'Cao B?ng',
        '??ng Nai',       '??ng Thap',         'Gia Lai',          'Ha Nam',
          'Ha N?i',       'H?i D??ng',       'H?i Phong',       'H?u Giang',
     'Ho Chi Minh',        'Hoa Binh',        'H?ng Yen',      'Kien Giang',
         'Kon Tum',        'Lam ??ng',        'L?ng S?n',         'Long An',
        'Nam ??nh',         'Ngh? An',       'Ninh Binh',      'Ninh Thu?n',
       'Qu?ng Nam',      'Qu?ng Ninh',       'Soc Tr?ng',          'S?n La',
        'Tay Ninh',       'Thai Binh',     'Thai Nguyen',       'Thanh Hoa',
  'Th?a Thien Hu?',      'Ti?n Giang',        'Tra Vinh',       'V?nh Long',
 'Ba R?a?V?ng Tau',        'B?c Lieu',       'Binh ??nh',          'Ca Mau',
         '?a N?ng',         '??k L?k',        '??k Nong',

In [25]:
# Mapping dictionary to replace corrupted names with clean English names
province_fixes_english = {
    'Ha T?nh': 'Ha Tinh',
    'Qu?ng Binh': 'Quang Binh',
    'An Giang': 'An Giang',
    'B?c Giang': 'Bac Giang',
    'B?c K?n': 'Bac Kan',
    'B?c Ninh': 'Bac Ninh',
    'B?n Tre': 'Ben Tre',
    'Binh D??ng': 'Binh Duong',
    'Binh Ph??c': 'Binh Phuoc',
    'Binh Thu?n': 'Binh Thuan',
    'C?n Th?': 'Can Tho',
    'Cao B?ng': 'Cao Bang',
    '??ng Nai': 'Dong Nai',
    '??ng Thap': 'Dong Thap',
    'Gia Lai': 'Gia Lai',
    'Ha Nam': 'Ha Nam',
    'Ha N?i': 'Ha Noi',
    'H?i D??ng': 'Hai Duong',
    'H?i Phong': 'Hai Phong',
    'H?u Giang': 'Hau Giang',
    'Ho Chi Minh': 'Ho Chi Minh',
    'Hoa Binh': 'Hoa Binh',
    'H?ng Yen': 'Hung Yen',
    'Kien Giang': 'Kien Giang',
    'Kon Tum': 'Kon Tum',
    'Lam ??ng': 'Lam Dong',
    'L?ng S?n': 'Lang Son',
    'Long An': 'Long An',
    'Nam ??nh': 'Nam Dinh',
    'Ngh? An': 'Nghe An',
    'Ninh Binh': 'Ninh Binh',
    'Ninh Thu?n': 'Ninh Thuan',
    'Qu?ng Nam': 'Quang Nam',
    'Qu?ng Ninh': 'Quang Ninh',
    'Soc Tr?ng': 'Soc Trang',
    'S?n La': 'Son La',
    'Tay Ninh': 'Tay Ninh',
    'Thai Binh': 'Thai Binh',
    'Thai Nguyen': 'Thai Nguyen',
    'Thanh Hoa': 'Thanh Hoa',
    'Th?a Thien Hu?': 'Thua Thien Hue',
    'Ti?n Giang': 'Tien Giang',
    'Tra Vinh': 'Tra Vinh',
    'V?nh Long': 'Vinh Long',
    'Ba R?a?V?ng Tau': 'Ba Ria - Vung Tau',
    'B?c Lieu': 'Bac Lieu',
    'Binh ??nh': 'Binh Dinh',
    'Ca Mau': 'Ca Mau',
    '?a N?ng': 'Da Nang',
    '??k L?k': 'Dak Lak',
    '??k Nong': 'Dak Nong',
    '?i?n Bien': 'Dien Bien',
    'Ha Giang': 'Ha Giang',
    'Khanh Hoa': 'Khanh Hoa',
    'Lai Chau': 'Lai Chau',
    'Lao Cai': 'Lao Cai',
    'Phu Th?': 'Phu Tho',
    'Phu Yen': 'Phu Yen',
    'Qu?ng Ngai': 'Quang Ngai',
    'Qu?ng Tr?': 'Quang Tri',
    'Tuyen Quang': 'Tuyen Quang',
    'V?nh Phuc': 'Vinh Phuc',
    'Yen Bai': 'Yen Bai',
    'Con ??o': 'Con Dao'
}

df_veg['province_name'] = df_veg['province_name'].replace(province_fixes_english)

In [26]:
df_veg.head()

,province_name,year,season,s2_n_images,NDVI_mean,NDVI_min,NDVI_max,NDVI_range,EVI_mean,EVI_min,EVI_max,EVI_range,LSWI_mean,LSWI_min,LSWI_max,LSWI_range
25,Hà Tĩnh,2015,autumn,1,0.689152,0.689152,0.689152,0.000000,0.503870,0.503870,0.503870,0.00000,0.242180,0.242180,0.242180,0.000000
46,Quảng Bình,2015,autumn,1,0.723619,0.723619,0.723619,0.000000,0.544738,0.544738,0.544738,0.00000,0.186920,0.186920,0.186920,0.000000
64,An Giang,2015,winter,48,0.377502,0.347905,0.410787,0.062882,0.278057,0.256314,0.305074,0.04876,0.324715,0.305206,0.340197,0.034992
66,Bắc Giang,2015,winter,48,0.474282,0.474256,0.474307,0.000051,0.309557,0.309547,0.309566,0.00002,0.108840,0.108832,0.108847,0.000015
67,Bắc Kạn,2015,winter,48,0.629547,0.629547,0.629547,0.000000,0.456883,0.456883,0.456883,0.00000,0.180317,0.180317,0.180317,0.000000


In [27]:
selected_columns = [
    "province_name",
    "year",
    "season",
    "NDVI_mean",
    "NDVI_max",
    "EVI_mean",
    "EVI_max",
    "LSWI_mean",
    "LSWI_min"
]

# Keep only the selected columns
df_selected = df[selected_columns].copy()

In [31]:
# Load datasets
primary = pd.read_parquet("../data/vietnam_rice_dataset_2015_2025.parquet")

# Columns used to match records
keys = ["province_name", "year", "season"]

# Normalize join keys to avoid mismatches due to whitespace
for df in (primary, df_selected):
    df["province_name"] = df["province_name"].astype(str).str.strip()
    df["season"] = df["season"].astype(str).str.strip()
    df["year"] = pd.to_numeric(df["year"], errors="raise").astype(int)

# Check for duplicate matching keys
print("Primary duplicate keys:", primary.duplicated(keys).sum())
print("Sentinel duplicate keys:", df_selected.duplicated(keys).sum())

# Ensure one row per province-year-season before merging
if primary.duplicated(keys).any():
    raise ValueError("Primary dataset has duplicate province-year-season keys.")

if df_selected.duplicated(keys).any():
    raise ValueError("Sentinel dataset has duplicate province-year-season keys.")

# Satellite features to add
satellite_features = [
    "NDVI_mean",
    "NDVI_max",
    "EVI_mean",
    "EVI_max",
    "LSWI_mean",
    "LSWI_min"
]

Primary duplicate keys: 0
Sentinel duplicate keys: 0


In [33]:
# Left join preserves all 16 years in the primary dataset
merged = primary.merge(
    df_selected[keys + satellite_features],
    on=keys,
    how="left",
    validate="one_to_one",
    indicator=True
)

# Check matching results
print("\nMerge results:")
print(merged["_merge"].value_counts())

print("\nRows with missing satellite features:")
print(merged[satellite_features].isna().any(axis=1).sum())

# Remove the temporary merge indicator
merged.drop(columns="_merge", inplace=True)

print("\nMerged dataset shape:", merged.shape)
print(merged.head())


Merge results:
_merge
left_only     1409
both           105
right_only       0
Name: count, dtype: int64

Rows with missing satellite features:
1417

Merged dataset shape: (1514, 24)
   VH_VV_ratio_mean     VH_max    VH_mean     VH_min  VH_range    VV_max  \
0         -6.379569 -13.731947 -17.018732 -21.186979  7.455031 -7.076649   
1         -6.767354 -13.347810 -15.764152 -18.815880  5.468069 -6.270434   
2         -7.440613 -13.339282 -17.567925 -21.703117  8.363835 -6.287126   
3         -6.479291 -13.488159 -16.793822 -21.734085  8.245926 -6.914853   
4         -6.519452 -13.820779 -16.456258 -19.896297  6.075518 -6.937031   

     VV_mean     VV_min  VV_range  precip_total_mm  ...  yield_ton_ha  \
0 -10.639163 -15.221711  8.145062       184.313428  ...          6.71   
1  -8.996798 -12.147528  5.877095       684.562912  ...          4.69   
2 -10.127312 -14.305305  8.018180       787.565689  ...          3.53   
3 -10.314531 -14.492940  7.578087        86.911599  ...          6.

# Keep only primary records that have a matching
# province, year, and season in the Sentinel-2 dataset.
overlap = primary.merge(
    sentinel[keys].drop_duplicates(),
    on=keys,
    how="inner",
    validate="one_to_one"
)

# Merge the actual satellite features
merged = overlap.merge(
    sentinel,
    on=keys,
    how="left",
    validate="one_to_one"
)

merged.to_csv("paddy_yield_overlapping_years.csv", index=False)

print("Overlapping dataset shape:", merged.shape)

In [34]:
merged.to_parquet("paddy_yield_sentinel2.parquet", index=False)

print("File saved as paddy_yield_sentinel2.parquet!")

File saved as paddy_yield_sentinel2.parquet!
